In [1]:
import sys
import subprocess
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message=".*not chronologically ordered.*")

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
sys.path.insert(0, str(project_root))

print(f"project_root exists: {project_root.exists()}")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

project_root exists: True


/home/nvidia/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [2]:
# ============================================================
# 6.2 setup: confirm we have paired NFS data for GAT vs classical
# ML across the 6 paradigms, ready for Wilcoxon testing
# ============================================================
gat_nfs = pd.read_csv(project_root / "results_gat" / "nfs" / "gat_nfs.csv")
rf_nfs = pd.read_csv(project_root / "results_stage4_classical_ml" / "nfs" / "random_forest_nfs.csv")
xgb_nfs = pd.read_csv(project_root / "results_stage4_classical_ml" / "nfs" / "xgboost_nfs.csv")

print(f"GAT NFS rows: {len(gat_nfs)}")
print(f"RF NFS rows: {len(rf_nfs)}")
print(f"XGBoost NFS rows: {len(xgb_nfs)}")
print(f"\nGAT columns: {list(gat_nfs.columns)}")
print(f"RF columns: {list(rf_nfs.columns)}")

GAT NFS rows: 60
RF NFS rows: 211
XGBoost NFS rows: 210

GAT columns: ['dataset_task', 'model', 'step', 'nfs', 'ci_low', 'ci_high', 'note']
RF columns: ['dataset_task', 'model', 'step', 'nfs', 'ci_low', 'ci_high', 'note']


In [3]:
# ============================================================
# Build the paired dataset: GAT NFS vs. best classical ML NFS,
# matched by (paradigm, step). Naming mismatch handled explicitly
# (GAT: single underscore, classical ML: double underscore) --
# same issue we hit and fixed earlier this project.
# ============================================================
gat_nfs["paradigm_norm"] = gat_nfs["dataset_task"].str.replace("ERP_CORE_", "ERP_CORE__", regex=False)

paradigms = ["ERP_CORE__P3", "ERP_CORE__MMN", "ERP_CORE__N2pc", "ERP_CORE__N400", "ERP_CORE__N170", "ERP_CORE__ERN"]
STEP_ORDER = ["5", "10", "20", "30", "50", "75", "100", "150", "200", "full"]

paired_rows = []
for paradigm in paradigms:
    for step in STEP_ORDER:
        gat_row = gat_nfs[(gat_nfs["paradigm_norm"] == paradigm) & (gat_nfs["step"].astype(str) == step)]
        rf_row = rf_nfs[(rf_nfs["dataset_task"] == paradigm) & (rf_nfs["step"].astype(str) == step)]
        xgb_row = xgb_nfs[(xgb_nfs["dataset_task"] == paradigm) & (xgb_nfs["step"].astype(str) == step)]

        gat_val = gat_row["nfs"].values[0] if len(gat_row) and pd.notna(gat_row["nfs"].values[0]) else None
        rf_val = rf_row["nfs"].values[0] if len(rf_row) and pd.notna(rf_row["nfs"].values[0]) else None
        xgb_val = xgb_row["nfs"].values[0] if len(xgb_row) and pd.notna(xgb_row["nfs"].values[0]) else None

        # "Best classical ML" = whichever of RF/XGBoost has a higher NFS at this step
        ml_candidates = [v for v in [rf_val, xgb_val] if v is not None]
        best_ml_val = max(ml_candidates) if ml_candidates else None

        if gat_val is not None and best_ml_val is not None:
            paired_rows.append({
                "paradigm": paradigm.replace("ERP_CORE__", ""),
                "step": step,
                "gat_nfs": gat_val,
                "best_ml_nfs": best_ml_val,
            })

paired_df = pd.DataFrame(paired_rows)
print(f"Total paired (paradigm, step) rows with both GAT and ML NFS: {len(paired_df)}")
print(paired_df.to_string(index=False))

Total paired (paradigm, step) rows with both GAT and ML NFS: 51
paradigm step   gat_nfs  best_ml_nfs
      P3    5  0.135596     0.021367
      P3   10  0.020717    -0.142996
      P3   20  0.108178     0.002684
      P3   30 -0.052655     0.089573
      P3   50  0.080897     0.228292
      P3   75 -0.089706     0.268618
      P3  100 -0.240778     0.268618
      P3  150  0.154812     0.268618
      P3  200 -0.166165     0.268618
      P3 full  0.019376    -0.077546
     MMN    5 -0.014044    -0.092665
     MMN   10  0.098413     0.112637
     MMN   20  0.231307    -0.077867
     MMN   30  0.141365     0.012026
     MMN   50  0.385452    -0.131114
     MMN   75 -0.222196    -0.049869
     MMN  100  0.406466    -0.040320
     MMN  150 -0.146874    -0.106980
     MMN  200  0.121332    -0.019153
     MMN full -0.313903    -0.163234
    N2pc    5 -0.064691     0.203150
    N2pc   10 -0.332757    -0.010930
    N2pc   20  0.184664    -0.184863
    N2pc   30  0.213291    -0.062280
    N2pc   

In [4]:
# ============================================================
# 6.2: Wilcoxon signed-rank test, GAT NFS vs. best classical ML
# NFS, per paradigm, with Bonferroni correction across paradigms
# ============================================================
from scipy.stats import wilcoxon

results = []
for paradigm in paired_df["paradigm"].unique():
    subset = paired_df[paired_df["paradigm"] == paradigm]
    gat_vals = subset["gat_nfs"].values
    ml_vals = subset["best_ml_nfs"].values

    if len(gat_vals) < 5:
        results.append({
            "paradigm": paradigm,
            "n_pairs": len(gat_vals),
            "mean_diff (GAT - ML)": np.mean(gat_vals - ml_vals) if len(gat_vals) > 0 else np.nan,
            "raw_p": np.nan,
            "note": "Insufficient pairs for a meaningful test (n<5)",
        })
        continue

    try:
        stat, p_val = wilcoxon(gat_vals, ml_vals)
        results.append({
            "paradigm": paradigm,
            "n_pairs": len(gat_vals),
            "mean_diff (GAT - ML)": np.mean(gat_vals - ml_vals),
            "raw_p": p_val,
            "note": "",
        })
    except Exception as e:
        results.append({
            "paradigm": paradigm,
            "n_pairs": len(gat_vals),
            "mean_diff (GAT - ML)": np.mean(gat_vals - ml_vals),
            "raw_p": np.nan,
            "note": f"Test failed: {e}",
        })

results_df = pd.DataFrame(results)

# Bonferroni correction across the paradigms that were actually tested
n_tests = results_df["raw_p"].notna().sum()
results_df["bonferroni_p"] = (results_df["raw_p"] * n_tests).clip(upper=1.0)
results_df["significant (α=0.05, corrected)"] = results_df["bonferroni_p"] < 0.05

print(f"Number of paradigms with a valid test: {n_tests}")
print(results_df.to_string(index=False))

out_path = project_root / "results_gat" / "Stage2_vs_Stage1_Wilcoxon.csv"
results_df.to_csv(out_path, index=False)
print(f"\n[Saved] {out_path}")

Number of paradigms with a valid test: 5
paradigm  n_pairs  mean_diff (GAT - ML)    raw_p                                           note  bonferroni_p  significant (α=0.05, corrected)
      P3       10             -0.122557 0.193359                                                     0.966797                            False
     MMN       10              0.124386 0.275391                                                     1.000000                            False
    N2pc       10              0.010370 1.000000                                                     1.000000                            False
    N400       10             -0.138212 0.105469                                                     0.527344                            False
    N170        1              0.143005      NaN Insufficient pairs for a meaningful test (n<5)           NaN                            False
     ERN       10             -0.229935 0.001953                                                     

In [5]:
# ============================================================
# 6.3: Finalize the minimum-trial table. We already have a
# draft from earlier (MinTrialTable_draft.csv) -- confirm it's
# current and correct given everything since (including the
# Wilcoxon results), then save as the official deliverable.
# ============================================================
draft_path = project_root / "results_gat" / "MinTrialTable_draft.csv"
if draft_path.exists():
    draft = pd.read_csv(draft_path)
    print(f"Existing draft: {len(draft)} rows")
    print(draft.to_string(index=False))
else:
    print("Draft not found -- will need to rebuild from scratch")

Existing draft: 18 rows
paradigm         model min_trial_count                                                                                           note
      P3           GAT   NOT CONVERGED No convergence point found; overall decreasing trend (Δ=-0.1162), never plateaus under 2% rule
      P3 Random Forest              30                                                     Converged (gain=-0.0127 over steps 30->75)
      P3       XGBoost               5                                                      Converged (gain=-0.0187 over steps 5->20)
     MMN           GAT   NOT CONVERGED No convergence point found; overall decreasing trend (Δ=-0.2999), never plateaus under 2% rule
     MMN Random Forest              75                                                     Converged (gain=0.0029 over steps 75->150)
     MMN       XGBoost               5                                                       Converged (gain=0.0148 over steps 5->20)
    N2pc           GAT              20

In [6]:
# ============================================================
# 6.3: Finalize the minimum-trial table -- cross-reference with
# the 6.2 Wilcoxon results, then save as the official deliverable
# ============================================================
wilcoxon_results = pd.read_csv(project_root / "results_gat" / "Stage2_vs_Stage1_Wilcoxon.csv")
sig_lookup = wilcoxon_results.set_index("paradigm")["significant (α=0.05, corrected)"].to_dict()

draft["gat_vs_ml_statistically_significant"] = draft["paradigm"].map(sig_lookup)
draft["gat_vs_ml_statistically_significant"] = draft["gat_vs_ml_statistically_significant"].fillna("Not tested (insufficient data)")

final_path = project_root / "results_gat" / "MinTrialTable_FINAL.csv"
draft.to_csv(final_path, index=False)

print(f"[Saved] {final_path}\n")
pd.set_option("display.max_colwidth", 60)
print(draft.to_string(index=False))

n_converged = (draft["min_trial_count"] != "NOT CONVERGED").sum()
print(f"\n{n_converged} of {len(draft)} entries show a clear convergence point.")
print(f"Only ERN shows a statistically significant GAT-vs-ML difference (favoring classical ML).")

[Saved] /lp-dev/23BRS1328/erp_trial_min/results_gat/MinTrialTable_FINAL.csv

paradigm         model min_trial_count                                                                                           note  gat_vs_ml_statistically_significant
      P3           GAT   NOT CONVERGED No convergence point found; overall decreasing trend (Δ=-0.1162), never plateaus under 2% rule                                False
      P3 Random Forest              30                                                     Converged (gain=-0.0127 over steps 30->75)                                False
      P3       XGBoost               5                                                      Converged (gain=-0.0187 over steps 5->20)                                False
     MMN           GAT   NOT CONVERGED No convergence point found; overall decreasing trend (Δ=-0.2999), never plateaus under 2% rule                                False
     MMN Random Forest              75                              

In [7]:
# ============================================================
# 6.1 setup: check what raw epoch data is available for building
# a per-variant reference topography for ds002680
# ============================================================
from dl_prep.explicit_epoch_loader import load_explicit, CONDITION_SPEC

# Confirm all 13 gonogo variants and their real structure
gonogo_keys = [k for k in CONDITION_SPEC if k.startswith("DS002680_gonogo")]
print(f"Found {len(gonogo_keys)} gonogo variants: {sorted(gonogo_keys)}")

# Load one variant to inspect real shape/structure before building the pipeline
X, y, subject_ids, ch_names, sfreq = load_explicit(
    project_root / "results_batch", "DS002680", "gonogo_1"
)
print(f"\ngonogo_1: X={X.shape}, subjects={len(set(subject_ids))}, sfreq={sfreq}")
print(f"Channel names: {ch_names}")
print(f"Channel count: {len(ch_names)}")

Found 13 gonogo variants: ['DS002680_gonogo_1', 'DS002680_gonogo_10', 'DS002680_gonogo_11', 'DS002680_gonogo_12', 'DS002680_gonogo_13', 'DS002680_gonogo_2', 'DS002680_gonogo_3', 'DS002680_gonogo_4', 'DS002680_gonogo_5', 'DS002680_gonogo_6', 'DS002680_gonogo_7', 'DS002680_gonogo_8', 'DS002680_gonogo_9']
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]

gonogo_1: X=(649, 31, 1001), subjects=10, sfreq=1000.0
Channel names: ['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'PZ', "T5'", "T6'", "O1'", "O2'", 'P3"', 'P4"', 'PZ"', 'OZ', 'I', 'CB1"', 'CB2"', 'CB1', 'CB2']
Channel count: 31


In [8]:
# ============================================================
# Verify channel naming is CONSISTENT across all 13 gonogo
# variants before building anything on top of it -- an
# inconsistency here would corrupt any reference topography
# ============================================================
all_channel_sets = {}
for key in sorted(gonogo_keys):
    dataset, task = key.split("_", 1)
    try:
        X, y, subject_ids, ch_names, sfreq = load_explicit(
            project_root / "results_batch", dataset, task
        )
        all_channel_sets[key] = (tuple(ch_names), len(ch_names), sfreq)
    except Exception as e:
        print(f"{key}: ERROR - {e}")

# Check if all variants share the identical channel list and order
reference_channels = all_channel_sets[list(all_channel_sets.keys())[0]][0]
all_match = all(v[0] == reference_channels for v in all_channel_sets.values())

print(f"All 13 variants have IDENTICAL channel names/order: {all_match}")
for key, (chans, n, sf) in all_channel_sets.items():
    print(f"{key}: n_channels={n}, sfreq={sf}, matches_reference={chans == reference_channels}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
[Explicit Loader] DS002680_gonogo_10: X=(902, 31, 1001), y=(902,), classes=[0 1], subjects=12, class_balance=[471, 431]
[Explicit Loader] DS002680_gonogo_11: X=(823, 31, 1001), y=(823,), classes=[0 1], subjects=12, class_balance=[432, 391]
[Explicit Loader] DS002680_gonogo_12: X=(728, 31, 1001), y=(728,), classes=[0 1], subjects=11, class_balance=[372, 356]
[Explicit Loader] DS002680_gonogo_13: X=(686, 31, 1001), y=(686,), classes=[0 1], subjects=11, class_balance=[341, 345]
[Explicit Loader] DS002680_gonogo_2: X=(642, 31, 1001), y=(642,), classes=[0 1], subjects=11, class_balance=[318, 324]
[Explicit Loader] DS002680_gonogo_3: X=(817, 31, 1001), y=(817,), classes=[0 1], subjects=12, class_balance=[413, 404]
[Explicit Loader] DS002680_gonogo_4: X=(783, 31, 1001), y=(783,), classes=[0 1], subjects=11, class_balance=[419, 364]
[Explicit Loader] DS002680_gonogo_5: X=(621,

In [9]:
# ============================================================
# 6.1: Build a grand-average reference topography for ONE
# variant first (gonogo_1) as a test case, before scaling to
# all 13 -- same small-test-first discipline as always.
# ============================================================
def build_grand_average_topography(X, y, sfreq, window_ms=(200, 500), epoch_start_ms=-200):
    """
    X: (n_trials, n_channels, n_times)
    y: (n_trials,) binary labels
    Returns: per-channel topography = mean(condition_1) - mean(condition_0)
             amplitude in the given time window, averaged across all trials.
    """
    n_times = X.shape[2]
    start_idx = int((window_ms[0] - epoch_start_ms) / 1000 * sfreq)
    end_idx = int((window_ms[1] - epoch_start_ms) / 1000 * sfreq)
    start_idx = max(0, start_idx)
    end_idx = min(n_times, end_idx)

    window_mean = X[:, :, start_idx:end_idx].mean(axis=2)  # (n_trials, n_channels)

    cond0_mean = window_mean[y == 0].mean(axis=0)  # (n_channels,)
    cond1_mean = window_mean[y == 1].mean(axis=0)

    topo = cond1_mean - cond0_mean
    return topo, (start_idx, end_idx)


X, y, subject_ids, ch_names, sfreq = load_explicit(
    project_root / "results_batch", "DS002680", "gonogo_1"
)
topo, (s, e) = build_grand_average_topography(X, y, sfreq)
print(f"Time window indices: {s} to {e} (out of {X.shape[2]} total)")
print(f"Topography shape: {topo.shape}")
print(f"Topography values: min={topo.min():.6f}, max={topo.max():.6f}, std={topo.std():.6f}")
print(f"\nPer-channel values:")
for ch, val in zip(ch_names, topo):
    print(f"  {ch}: {val:.6f}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Time window indices: 400 to 700 (out of 1001 total)
Topography shape: (31,)
Topography values: min=-0.000002, max=0.000002, std=0.000001

Per-channel values:
  FP1: 0.000000
  FP2: 0.000000
  F3: 0.000001
  F4: -0.000000
  C3: -0.000001
  C4: -0.000001
  P3: -0.000002
  P4: -0.000000
  O1: -0.000001
  O2: 0.000002
  F7: -0.000001
  F8: -0.000000
  T3: -0.000001
  T4: 0.000001
  T5: 0.000001
  T6: 0.000001
  FZ: 0.000001
  PZ: -0.000002
  T5': -0.000001
  T6': 0.000001
  O1': -0.000000
  O2': 0.000001
  P3": -0.000001
  P4": 0.000000
  PZ": -0.000002
  OZ: 0.000001
  I: -0.000000
  CB1": 0.000002
  CB2": -0.000000
  CB1: -0.000000
  CB2: -0.000000


In [10]:
# ============================================================
# Sanity check: does this topography show a plausible spatial
# pattern (fronto-central emphasis, consistent with known
# go/no-go N2/P3 effects), or does it look like pure noise?
# ============================================================
import numpy as np

frontocentral = ["FZ", "F3", "F4", "C3", "C4"]
posterior = ["O1", "O2", "OZ", "PZ", "P3", "P4"]

fc_idx = [ch_names.index(ch) for ch in frontocentral if ch in ch_names]
post_idx = [ch_names.index(ch) for ch in posterior if ch in ch_names]

fc_mean_abs = np.abs(topo[fc_idx]).mean()
post_mean_abs = np.abs(topo[post_idx]).mean()

print(f"Mean |amplitude| at fronto-central sites: {fc_mean_abs:.6f}")
print(f"Mean |amplitude| at posterior sites: {post_mean_abs:.6f}")
print(f"Ratio (fronto-central / posterior): {fc_mean_abs / post_mean_abs:.2f}")
print(f"\n(A ratio meaningfully > 1 would be consistent with a plausible go/no-go effect;")
print(f"a ratio near 1 or < 1 would suggest this specific window/reference may not be capturing a clean spatial effect)")

Mean |amplitude| at fronto-central sites: 0.000001
Mean |amplitude| at posterior sites: 0.000001
Ratio (fronto-central / posterior): 0.62

(A ratio meaningfully > 1 would be consistent with a plausible go/no-go effect;
a ratio near 1 or < 1 would suggest this specific window/reference may not be capturing a clean spatial effect)


In [11]:
# ============================================================
# Check 1: is this even a statistically real difference, or
# plausibly just noise given only 10 subjects?
# ============================================================
from scipy.stats import ttest_ind

start_idx, end_idx = 400, 700
window_mean = X[:, :, start_idx:end_idx].mean(axis=2)

# Per-subject averaging first (correct unit of analysis -- avoids
# pseudo-replication from treating trials as independent subjects)
subject_diffs = []
for subj in sorted(set(subject_ids)):
    subj_mask = subject_ids == subj
    subj_y = y[subj_mask]
    subj_window = window_mean[subj_mask]
    if len(set(subj_y)) < 2:
        continue
    diff = subj_window[subj_y == 1].mean(axis=0) - subj_window[subj_y == 0].mean(axis=0)
    subject_diffs.append(diff)

subject_diffs = np.array(subject_diffs)  # (n_subjects, n_channels)
print(f"Per-subject difference array shape: {subject_diffs.shape}")

# One-sample t-test per channel: is the across-subject mean difference
# significantly different from zero?
from scipy.stats import ttest_1samp
t_stats, p_vals = ttest_1samp(subject_diffs, 0, axis=0)
n_significant = (p_vals < 0.05).sum()
print(f"Channels with a statistically significant (p<0.05, uncorrected) go/no-go difference: {n_significant} / 31")
print(f"Fronto-central p-values: {dict(zip(frontocentral, p_vals[fc_idx]))}")

# ============================================================
# Check 2: is this pattern consistent across a couple more variants?
# ============================================================
for variant in ["gonogo_3", "gonogo_7"]:
    Xv, yv, sidv, chv, sfv = load_explicit(project_root / "results_batch", "DS002680", variant)
    topo_v, _ = build_grand_average_topography(Xv, yv, sfv)
    fc_v = np.abs(topo_v[fc_idx]).mean()
    post_v = np.abs(topo_v[post_idx]).mean()
    print(f"\n{variant}: fronto-central={fc_v:.6f}, posterior={post_v:.6f}, ratio={fc_v/post_v:.2f}")

Per-subject difference array shape: (10, 31)
Channels with a statistically significant (p<0.05, uncorrected) go/no-go difference: 2 / 31
Fronto-central p-values: {'FZ': np.float64(0.3942678824573357), 'F3': np.float64(0.18318128314074708), 'F4': np.float64(0.9020731425760046), 'C3': np.float64(0.9182764231702832), 'C4': np.float64(0.3337847681316308)}
[Explicit Loader] DS002680_gonogo_3: X=(817, 31, 1001), y=(817,), classes=[0 1], subjects=12, class_balance=[413, 404]

gonogo_3: fronto-central=0.000007, posterior=0.000007, ratio=1.05
[Explicit Loader] DS002680_gonogo_7: X=(836, 31, 1001), y=(836,), classes=[0 1], subjects=12, class_balance=[434, 402]

gonogo_7: fronto-central=0.000001, posterior=0.000001, ratio=1.38


In [12]:
# ============================================================
# Principled alternative: per-subject peak-latency windowing,
# not a scan-for-best-result. Standard ERP technique that
# accounts for real subject-to-subject latency variability.
# ============================================================
def build_peak_latency_topography(X, y, sfreq, search_ms=(200, 600), epoch_start_ms=-200, window_half_width_ms=50):
    """
    For each subject, find the time sample where |go - nogo| difference
    peaks (Global Field Power of the difference wave) within the search
    window, then average a small window around that peak. This is
    standard adaptive-window ERP practice, not a post-hoc best-fit scan.
    """
    n_times = X.shape[2]
    search_start = int((search_ms[0] - epoch_start_ms) / 1000 * sfreq)
    search_end = int((search_ms[1] - epoch_start_ms) / 1000 * sfreq)
    half_width = int(window_half_width_ms / 1000 * sfreq)

    subject_diffs = []
    peak_latencies = []
    for subj in sorted(set(subject_ids)):
        subj_mask = subject_ids == subj
        subj_y = y[subj_mask]
        if len(set(subj_y)) < 2:
            continue
        subj_X = X[subj_mask]

        cond0 = subj_X[subj_y == 0].mean(axis=0)  # (n_channels, n_times)
        cond1 = subj_X[subj_y == 1].mean(axis=0)
        diff_wave = cond1 - cond0  # (n_channels, n_times)

        # Global field power of the difference wave -- standard way to
        # find a subject's own peak difference latency, channel-agnostic
        gfp = diff_wave.std(axis=0)
        search_gfp = gfp[search_start:search_end]
        peak_idx = search_start + np.argmax(search_gfp)
        peak_latencies.append(peak_idx)

        w_start = max(0, peak_idx - half_width)
        w_end = min(n_times, peak_idx + half_width)
        subject_diffs.append(diff_wave[:, w_start:w_end].mean(axis=1))

    subject_diffs = np.array(subject_diffs)
    topo = subject_diffs.mean(axis=0)
    return topo, subject_diffs, peak_latencies


X, y, subject_ids, ch_names, sfreq = load_explicit(
    project_root / "results_batch", "DS002680", "gonogo_1"
)
topo2, subject_diffs2, peaks = build_peak_latency_topography(X, y, sfreq)

peak_times_ms = [(p / sfreq * 1000) - 200 for p in peaks]
print(f"Per-subject peak latencies (ms post-stimulus): {[f'{t:.0f}' for t in peak_times_ms]}")

from scipy.stats import ttest_1samp
t_stats2, p_vals2 = ttest_1samp(subject_diffs2, 0, axis=0)
n_sig2 = (p_vals2 < 0.05).sum()
print(f"\nChannels with p<0.05 (uncorrected): {n_sig2} / 31")
print(f"Fronto-central p-values: {dict(zip(frontocentral, p_vals2[fc_idx]))}")

fc_mean2 = np.abs(topo2[fc_idx]).mean()
post_mean2 = np.abs(topo2[post_idx]).mean()
print(f"\nFronto-central/posterior ratio: {fc_mean2/post_mean2:.2f}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Per-subject peak latencies (ms post-stimulus): ['211', '511', '599', '300', '489', '558', '420', '560', '399', '243']

Channels with p<0.05 (uncorrected): 1 / 31
Fronto-central p-values: {'FZ': np.float64(0.5563402719954613), 'F3': np.float64(0.4883484492178465), 'F4': np.float64(0.7284141882264675), 'C3': np.float64(0.577415564382709), 'C4': np.float64(0.24225898775394264)}

Fronto-central/posterior ratio: 0.77


In [13]:
# ============================================================
# Option B: pool subjects across multiple gonogo variants to
# test whether more power reveals a real effect -- diagnostic
# only, not building a final reference from this.
# ============================================================
all_pooled_diffs = []
all_pooled_subject_ids = []

for variant_key in sorted(gonogo_keys):
    dataset, task = variant_key.split("_", 1)
    Xv, yv, sidv, chv, sfv = load_explicit(project_root / "results_batch", dataset, task)

    window_mean_v = Xv[:, :, 400:700].mean(axis=2)  # same 200-500ms window as before

    for subj in sorted(set(sidv)):
        subj_mask = sidv == subj
        subj_y = yv[subj_mask]
        if len(set(subj_y)) < 2:
            continue
        subj_window = window_mean_v[subj_mask]
        diff = subj_window[subj_y == 1].mean(axis=0) - subj_window[subj_y == 0].mean(axis=0)
        all_pooled_diffs.append(diff)
        # Unique subject identifier across variants (same physical subject
        # may appear in multiple variants, but treat each variant's
        # instance as one independent observation for this power check)
        all_pooled_subject_ids.append(f"{variant_key}_{subj}")

all_pooled_diffs = np.array(all_pooled_diffs)
print(f"Total pooled (subject, variant) observations: {all_pooled_diffs.shape[0]}")
print(f"(compare: single-variant test used only 10 observations)")

from scipy.stats import ttest_1samp
t_stats_pooled, p_vals_pooled = ttest_1samp(all_pooled_diffs, 0, axis=0)
n_sig_pooled = (p_vals_pooled < 0.05).sum()
print(f"\nChannels with p<0.05 (uncorrected), pooled across all 13 variants: {n_sig_pooled} / 31")
print(f"Fronto-central p-values: {dict(zip(frontocentral, p_vals_pooled[fc_idx]))}")

# Bonferroni-corrected too, since we have real power to spare here
n_sig_corrected = (p_vals_pooled < 0.05 / 31).sum()
print(f"Channels surviving Bonferroni correction (p < {0.05/31:.4f}): {n_sig_corrected} / 31")

pooled_topo = all_pooled_diffs.mean(axis=0)
fc_pooled = np.abs(pooled_topo[fc_idx]).mean()
post_pooled = np.abs(pooled_topo[post_idx]).mean()
print(f"\nFronto-central/posterior ratio (pooled): {fc_pooled/post_pooled:.2f}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
[Explicit Loader] DS002680_gonogo_10: X=(902, 31, 1001), y=(902,), classes=[0 1], subjects=12, class_balance=[471, 431]
[Explicit Loader] DS002680_gonogo_11: X=(823, 31, 1001), y=(823,), classes=[0 1], subjects=12, class_balance=[432, 391]
[Explicit Loader] DS002680_gonogo_12: X=(728, 31, 1001), y=(728,), classes=[0 1], subjects=11, class_balance=[372, 356]
[Explicit Loader] DS002680_gonogo_13: X=(686, 31, 1001), y=(686,), classes=[0 1], subjects=11, class_balance=[341, 345]
[Explicit Loader] DS002680_gonogo_2: X=(642, 31, 1001), y=(642,), classes=[0 1], subjects=11, class_balance=[318, 324]
[Explicit Loader] DS002680_gonogo_3: X=(817, 31, 1001), y=(817,), classes=[0 1], subjects=12, class_balance=[413, 404]
[Explicit Loader] DS002680_gonogo_4: X=(783, 31, 1001), y=(783,), classes=[0 1], subjects=11, class_balance=[419, 364]
[Explicit Loader] DS002680_gonogo_5: X=(621,

In [14]:
# ============================================================
# Critical check: how many TRULY unique subjects are in the
# pooled sample, and does the effect survive when we correctly
# treat each unique human as ONE observation (not one per variant)?
# ============================================================

# First: how much repetition is really happening?
unique_true_subjects = set()
for pooled_id in all_pooled_subject_ids:
    true_subj = pooled_id.split("_")[-1]  # extract "sub-XXX" part
    unique_true_subjects.add(true_subj)

print(f"Pooled observations: {len(all_pooled_subject_ids)}")
print(f"TRUE unique subjects: {len(unique_true_subjects)}")
print(f"Average variants per subject: {len(all_pooled_subject_ids) / len(unique_true_subjects):.1f}")

# Correct approach: average each unique subject's diff-wave ACROSS
# all variants they appear in first, so each real human contributes
# exactly ONE independent observation to the test
subject_to_diffs = {}
for pooled_id, diff in zip(all_pooled_subject_ids, all_pooled_diffs):
    true_subj = pooled_id.split("_")[-1]
    subject_to_diffs.setdefault(true_subj, []).append(diff)

corrected_diffs = np.array([np.mean(diffs, axis=0) for diffs in subject_to_diffs.values()])
print(f"\nCorrected sample size (one observation per TRUE unique subject): {corrected_diffs.shape[0]}")

t_stats_corrected, p_vals_corrected = ttest_1samp(corrected_diffs, 0, axis=0)
n_sig_corrected2 = (p_vals_corrected < 0.05).sum()
print(f"Channels with p<0.05 (uncorrected), CORRECTED for true independence: {n_sig_corrected2} / 31")
print(f"Fronto-central p-values (corrected): {dict(zip(frontocentral, p_vals_corrected[fc_idx]))}")

Pooled observations: 144
TRUE unique subjects: 12
Average variants per subject: 12.0

Corrected sample size (one observation per TRUE unique subject): 12
Channels with p<0.05 (uncorrected), CORRECTED for true independence: 23 / 31
Fronto-central p-values (corrected): {'FZ': np.float64(0.00645752410577494), 'F3': np.float64(0.006237048306217426), 'F4': np.float64(0.0027787922155103075), 'C3': np.float64(0.19138930826175427), 'C4': np.float64(0.08117686644852906)}


In [15]:
# ============================================================
# Build the final, correctly-derived reference topography from
# the properly independent (per-true-subject) averaged data
# ============================================================
pooled_reference_topo = corrected_diffs.mean(axis=0)

fc_final = np.abs(pooled_reference_topo[fc_idx]).mean()
post_final = np.abs(pooled_reference_topo[post_idx]).mean()
print(f"Fronto-central/posterior ratio (corrected, final): {fc_final/post_final:.2f}")

print(f"\nFull channel topography:")
for ch, val, p in zip(ch_names, pooled_reference_topo, p_vals_corrected):
    marker = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    print(f"  {ch}: {val:.6f}  (p={p:.4f}) {marker}")

# Save this as the ds002680 pooled reference topography
save_path = project_root / "results_gat" / "ds002680_pooled_reference_topography.npy"
np.save(save_path, pooled_reference_topo)
print(f"\n[Saved] {save_path}")

Fronto-central/posterior ratio (corrected, final): 1.02

Full channel topography:
  FP1: 0.000006  (p=0.0029) **
  FP2: 0.000007  (p=0.0022) **
  F3: 0.000005  (p=0.0062) **
  F4: 0.000006  (p=0.0028) **
  C3: 0.000002  (p=0.1914) 
  C4: 0.000002  (p=0.0812) 
  P3: -0.000005  (p=0.0001) ***
  P4: -0.000005  (p=0.0002) ***
  O1: -0.000004  (p=0.0002) ***
  O2: -0.000003  (p=0.0017) **
  F7: 0.000006  (p=0.0007) ***
  F8: 0.000006  (p=0.0002) ***
  T3: 0.000004  (p=0.0002) ***
  T4: 0.000004  (p=0.0000) ***
  T5: -0.000002  (p=0.0260) *
  T6: -0.000002  (p=0.0935) 
  FZ: 0.000006  (p=0.0065) **
  PZ: -0.000004  (p=0.0080) **
  T5': 0.000002  (p=0.0390) *
  T6': 0.000002  (p=0.0354) *
  O1': -0.000004  (p=0.0037) **
  O2': -0.000003  (p=0.0081) **
  P3": -0.000005  (p=0.0004) ***
  P4": -0.000005  (p=0.0012) **
  PZ": -0.000006  (p=0.0000) ***
  OZ: -0.000003  (p=0.0071) **
  I: -0.000001  (p=0.1247) 
  CB1": -0.000001  (p=0.3628) 
  CB2": -0.000001  (p=0.4975) 
  CB1: -0.000001  (p=0.280

In [16]:
# ============================================================
# 6.1 final step: compute NFS for classical ML and GAT against
# the validated pooled reference topography, across all 13
# gonogo variants
# ============================================================
from stage4_classical_ml.nfs import bootstrap_nfs
import pickle

reference_topo = np.load(project_root / "results_gat" / "ds002680_pooled_reference_topography.npy")
print(f"Reference topography loaded: {reference_topo.shape}")

# --- Classical ML: check what feature-importance data exists for gonogo ---
# Recall classical ML doesn't currently save raw feature-importance
# arrays to disk (same limitation noted in Stage 1's original NFS work)
# -- confirm before assuming we can compute this side.
sample_model_dir = project_root / "results_stage4_classical_ml" / "random_forest" / "DS002680__gonogo_1"
print(f"\nClassical ML gonogo_1 dir exists: {sample_model_dir.exists()}")
if sample_model_dir.exists():
    print(f"Contents: {list(sample_model_dir.iterdir())[:10]}")

# --- GAT: check what we have for gonogo variants ---
gat_gonogo_dir = project_root / "results_gat" / "gat"
gonogo_gat_dirs = [d for d in gat_gonogo_dir.iterdir() if "gonogo" in d.name.lower()] if gat_gonogo_dir.exists() else []
print(f"\nGAT gonogo result directories found: {len(gonogo_gat_dirs)}")
print([d.name for d in gonogo_gat_dirs[:5]])

Reference topography loaded: (31,)

Classical ML gonogo_1 dir exists: True
Contents: [PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_5_folds.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/DS002680__gonogo_1_summary.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_10_folds.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_20_folds.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_30_folds.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_50_folds.csv'), PosixPath('/lp-dev/23BRS1328/erp_trial_min/results_stage4_classical_ml/random_forest/DS002680__gonogo_1/step_75_folds.csv'), PosixPath('/lp-dev/23BRS132

In [17]:
# ============================================================
# Check the existing classical ML NFS code structure before
# modifying it -- need to see exactly where feature importance
# is computed so we can add a save step there
# ============================================================
import inspect
from stage4_classical_ml import run_stage1_classical_ml as classical_driver

# Find the actual function that computes feature importance
src = inspect.getsource(classical_driver.calculate_and_save_nfs)
print(src[:3000])

def calculate_and_save_nfs(
    model_name: str,
    paradigm: str,
    step,
    data: ParadigmData,
    all_importances: list[np.ndarray],
) -> None:
    """
    Compute NFS when possible.

    NFS is skipped safely when:
    - No reference topography exists.
    - Model has no feature importance.
    - Feature/reference channel counts differ.
    - Feature block cannot be interpreted.
    """
    nfs_csv = NFS_DIR / f"{model_name}_nfs.csv"
    reference_name = PARADIGM_TO_NFS_NAME.get(paradigm)

    if reference_name is None:
        write_nfs_na(
            nfs_csv=nfs_csv,
            model_name=model_name,
            paradigm=paradigm,
            step=step,
            note="N/A: no NFS reference topography exists for this paradigm",
        )
        return

    if not all_importances:
        write_nfs_na(
            nfs_csv=nfs_csv,
            model_name=model_name,
            paradigm=paradigm,
            step=step,
            note="N/A: model does not provide usable 

In [18]:
# ============================================================
# Patch: save raw model-derived topography to disk BEFORE the
# reference-check early-return, so paradigms without a standard
# reference (all gonogo variants) still get their topography
# captured for later comparison against our custom pooled
# reference. Does not change any existing behavior for the 6
# ERP CORE paradigms -- purely additive.
# ============================================================
import inspect
classical_driver_path = project_root / "stage4_classical_ml" / "run_stage1_classical_ml.py"
text = classical_driver_path.read_text()

old = '''    nfs_csv = NFS_DIR / f"{model_name}_nfs.csv"
    reference_name = PARADIGM_TO_NFS_NAME.get(paradigm)

    if reference_name is None:
        write_nfs_na(
            nfs_csv=nfs_csv,
            model_name=model_name,
            paradigm=paradigm,
            step=step,
            note="N/A: no NFS reference topography exists for this paradigm",
        )
        return'''

new = '''    nfs_csv = NFS_DIR / f"{model_name}_nfs.csv"
    reference_name = PARADIGM_TO_NFS_NAME.get(paradigm)

    # ADDITIVE, non-disruptive save: capture the raw model-derived
    # topography for ANY paradigm with usable importances, even when
    # no standard reference exists (e.g. ds002680 gonogo variants).
    # This does not change behavior for the 6 ERP CORE paradigms below.
    if all_importances:
        try:
            mean_importance_raw = np.mean(np.stack(all_importances, axis=0), axis=0)
            topo_raw_result = importance_to_topography(mean_importance_raw, data.feature_blocks)
            topo_raw = topo_raw_result[0] if isinstance(topo_raw_result, tuple) else topo_raw_result
            topo_save_dir = OUT_ROOT / "raw_topographies" / model_name / paradigm
            topo_save_dir.mkdir(parents=True, exist_ok=True)
            np.save(topo_save_dir / f"step_{step}.npy", topo_raw)
        except Exception as exc:
            print(f"[TOPO SAVE WARNING] {model_name}/{paradigm}/step={step}: {exc}", flush=True)

    if reference_name is None:
        write_nfs_na(
            nfs_csv=nfs_csv,
            model_name=model_name,
            paradigm=paradigm,
            step=step,
            note="N/A: no NFS reference topography exists for this paradigm",
        )
        return'''

assert old in text, "Expected block not found -- paste the file again if this fails"
text = text.replace(old, new)
classical_driver_path.write_text(text)
print("[Patched] run_stage1_classical_ml.py -- raw topography now saved for all paradigms, including gonogo")

[Patched] run_stage1_classical_ml.py -- raw topography now saved for all paradigms, including gonogo


In [19]:
# ============================================================
# Delete markers ONLY for RF + XGBoost on the 13 gonogo variants,
# so re-running regenerates topographies without touching anything
# else (SVM markers, the other 9 paradigms, all untouched)
# ============================================================
marker_dir = project_root / "results_stage4_classical_ml" / "markers"

GONOGO_PARADIGMS = [f"DS002680__gonogo_{i}" for i in range(1, 14)]
STEPS = ["5", "10", "20", "30", "50", "75", "100", "150", "200", "full"]
MODELS = ["random_forest", "xgboost"]

to_delete = []
for model in MODELS:
    for paradigm in GONOGO_PARADIGMS:
        for step in STEPS:
            marker = marker_dir / f"{model}__{paradigm}__step{step}.done"
            if marker.exists():
                to_delete.append(marker)

print(f"Markers matching target set: {len(to_delete)} (expect up to 260: 2 models x 13 paradigms x 10 steps)")
print(f"Sample: {[m.name for m in to_delete[:5]]}")

Markers matching target set: 260 (expect up to 260: 2 models x 13 paradigms x 10 steps)
Sample: ['random_forest__DS002680__gonogo_1__step5.done', 'random_forest__DS002680__gonogo_1__step10.done', 'random_forest__DS002680__gonogo_1__step20.done', 'random_forest__DS002680__gonogo_1__step30.done', 'random_forest__DS002680__gonogo_1__step50.done']


In [20]:
# ============================================================
# Delete the 260 markers, then re-run classical ML specifically
# for these 13 paradigms in the background
# ============================================================
for m in to_delete:
    m.unlink()

remaining = [m for m in to_delete if m.exists()]
print(f"Deleted {len(to_delete)} markers. Remaining (should be 0): {len(remaining)}")

Deleted 260 markers. Remaining (should be 0): 0


In [21]:
# ============================================================
# Launch classical ML re-run, scoped to just gonogo paradigms,
# as a background process
# ============================================================
import subprocess

wrapper_code = '''
import sys
sys.path.insert(0, "/lp-dev/23BRS1328/erp_trial_min")
import warnings
warnings.filterwarnings("ignore")
from stage4_classical_ml import run_stage1_classical_ml as driver

GONOGO = [("DS002680", f"gonogo_{i}") for i in range(1, 14)]

for dataset, task in GONOGO:
    paradigm = f"{dataset}__{task}"
    print(f"\\n=== {paradigm} ===", flush=True)
    try:
        data = driver.ParadigmData(paradigm)
    except Exception as e:
        print(f"[ERROR] Could not load {paradigm}: {e}", flush=True)
        continue
    for model_name in ["random_forest", "xgboost"]:
        for step in driver.STEPS:
            try:
                driver.run_model_paradigm_step(model_name, paradigm, step, data)
            except AttributeError:
                print("[ERROR] run_model_paradigm_step not found -- check actual function name", flush=True)
                raise
            except Exception as e:
                print(f"[ERROR] {model_name}/{paradigm}/step={step}: {e}", flush=True)

print("\\nGonogo classical ML re-run complete.", flush=True)
'''

wrapper_path = project_root / "run_classical_gonogo_topo.py"
wrapper_path.write_text(wrapper_code)

log_path = project_root / "logs_classical_gonogo_topo.log"
with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        ["python3", str(wrapper_path)],
        cwd=str(project_root), stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=1279505
Log: /lp-dev/23BRS1328/erp_trial_min/logs_classical_gonogo_topo.log


In [22]:
import time
time.sleep(20)

import subprocess
result = subprocess.run(["ps", "-p", "1279505", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_classical_gonogo_topo.log").read_text()
print("=== Log ===")
print(log_text[-2000:] if log_text else "(no output yet)")

=== Process status ===
    PID %CPU STAT     ELAPSED
1279505  4.6 Zs         00:51

=== Log ===

=== DS002680__gonogo_1 ===
[ERROR] run_model_paradigm_step not found -- check actual function name
Traceback (most recent call last):
  File "/lp-dev/23BRS1328/erp_trial_min/run_classical_gonogo_topo.py", line 21, in <module>
    driver.run_model_paradigm_step(model_name, paradigm, step, data)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'stage4_classical_ml.run_stage1_classical_ml' has no attribute 'run_model_paradigm_step'



In [23]:
import inspect
from stage4_classical_ml import run_stage1_classical_ml as driver

# List all real functions in the module
funcs = [name for name, obj in inspect.getmembers(driver) if inspect.isfunction(obj)]
print("Functions in run_stage1_classical_ml.py:")
for f in funcs:
    print(f"  {f}")

Functions in run_stage1_classical_ml.py:
  append_row
  atomic_touch
  bootstrap_nfs
  calculate_and_save_nfs
  compute_metrics
  error_path
  get_nfs_reference_channel_count
  get_paradigms
  importance_to_topography
  load_reference_topography
  main
  marker_path
  run_one
  safe_step_name
  write_error
  write_nfs_na


In [24]:
import inspect
print(inspect.signature(driver.run_one))
print()
print(inspect.getsource(driver.run_one)[:1500])

(model_name: 'str', paradigm: 'str', step, data: 'ParadigmData') -> 'dict | None'

def calculate_and_save_nfs(
    model_name: str,
    paradigm: str,
    step,
    data: ParadigmData,
    all_importances: list[np.ndarray],
) -> None:
    """
    Compute NFS when possible.

    NFS is skipped safely when:
    - No reference topography exists.
    - Model has no feature importance.
    - Feature/reference channel counts differ.
    - Feature block cannot be interpreted.
    """
    nfs_csv = NFS_DIR / f"{model_name}_nfs.csv"
    reference_name = PARADIGM_TO_NFS_NAME.get(paradigm)

    # ADDITIVE, non-disruptive save: capture the raw model-derived
    # topography for ANY paradigm with usable importances, even when
    # no standard reference exists (e.g. ds002680 gonogo variants).
    # This does not change behavior for the 6 ERP CORE paradigms below.
    if all_importances:
        try:
            mean_importance_raw = np.mean(np.stack(all_importances, axis=0), axis=0)
            top

In [25]:
print(inspect.getsource(driver.run_one))

def calculate_and_save_nfs(
    model_name: str,
    paradigm: str,
    step,
    data: ParadigmData,
    all_importances: list[np.ndarray],
) -> None:
    """
    Compute NFS when possible.

    NFS is skipped safely when:
    - No reference topography exists.
    - Model has no feature importance.
    - Feature/reference channel counts differ.
    - Feature block cannot be interpreted.
    """
    nfs_csv = NFS_DIR / f"{model_name}_nfs.csv"
    reference_name = PARADIGM_TO_NFS_NAME.get(paradigm)

    # ADDITIVE, non-disruptive save: capture the raw model-derived
    # topography for ANY paradigm with usable importances, even when
    # no standard reference exists (e.g. ds002680 gonogo variants).
    # This does not change behavior for the 6 ERP CORE paradigms below.
    if all_importances:
        try:
            mean_importance_raw = np.mean(np.stack(all_importances, axis=0), axis=0)
            topo_raw_result = importance_to_topography(mean_importance_raw, data.feature_blocks)
 

In [26]:
# ============================================================
# Test run_one() directly on one small case, given we're having
# trouble reliably viewing its source -- test-first discipline
# ============================================================
try:
    data = driver.ParadigmData("DS002680__gonogo_1")
    result = driver.run_one("random_forest", "DS002680__gonogo_1", "5", data)
    print(f"run_one() result: {result}")
    print(f"Type: {type(result)}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

[START] model=random_forest paradigm=DS002680__gonogo_1 step=5
[DONE] model=random_forest paradigm=DS002680__gonogo_1 step=5 accuracy=0.5212 balanced_accuracy=0.5410 degenerate=0/10
run_one() result: {'dataset_task': 'DS002680__gonogo_1', 'model': 'random_forest', 'step': '5', 'n_folds': 10, 'n_degenerate_folds': 0, 'accuracy_mean': 0.5211954165547386, 'balanced_accuracy_mean': 0.5410424947393032, 'macro_f1_mean': 0.3891621489553129, 'auc_mean': nan}
Type: <class 'dict'>


In [27]:
# ============================================================
# Verify our topography-saving patch actually worked on this test run
# ============================================================
topo_path = project_root / "results_stage4_classical_ml" / "raw_topographies" / "random_forest" / "DS002680__gonogo_1" / "step_5.npy"
print(f"Topography file exists: {topo_path.exists()}")
if topo_path.exists():
    topo = np.load(topo_path)
    print(f"Shape: {topo.shape}")
    print(f"Values: min={topo.min():.4f}, max={topo.max():.4f}")

Topography file exists: False


In [28]:
import inspect
main_src = inspect.getsource(driver.main)
# Print just the part of main() that shows the calling order
lines = main_src.splitlines()
for i, line in enumerate(lines):
    if "run_one" in line or "calculate_and_save_nfs" in line or "all_importances" in line:
        print(f"{i}: {line}")

0: def run_one(
22:     all_importances: list[np.ndarray] = []
109:                     all_importances.append(importance)
188:     calculate_and_save_nfs(
193:         all_importances=all_importances,


In [29]:
import inspect
main_src = inspect.getsource(driver.main)
print(f"Total lines: {len(main_src.splitlines())}")
print(main_src)

Total lines: 209
def run_one(
    model_name: str,
    paradigm: str,
    step,
    data: ParadigmData,
) -> dict | None:
    """
    Run one model on one paradigm and one trial-count step.
    """
    marker = marker_path(model_name, paradigm, step)

    if marker.exists():
        return None

    print(
        f"[START] model={model_name} "
        f"paradigm={paradigm} "
        f"step={step}",
        flush=True,
    )

    fold_rows: list[dict] = []
    all_importances: list[np.ndarray] = []

    subjects = data.subjects()

    for fold_number, test_subject in enumerate(subjects, start=1):
        train_subjects = [
            subject
            for subject in subjects
            if subject != test_subject
        ]

        train_rows: list[int] = []

        for train_subject in train_subjects:
            if str(step) == "full":
                rows = data.all_row_indices_for_subject(
                    train_subject
                )
            else:
                row

In [30]:
# ============================================================
# Reload the module to pick up our file patch, then re-test
# ============================================================
import importlib
from stage4_classical_ml import run_stage1_classical_ml as driver
importlib.reload(driver)

# Delete the marker from our first test so it actually re-runs
marker = driver.marker_path("random_forest", "DS002680__gonogo_1", "5")
marker.unlink(missing_ok=True)

data = driver.ParadigmData("DS002680__gonogo_1")
result = driver.run_one("random_forest", "DS002680__gonogo_1", "5", data)
print(f"Result: {result}")

# Now check for the topography file again
topo_path = project_root / "results_stage4_classical_ml" / "raw_topographies" / "random_forest" / "DS002680__gonogo_1" / "step_5.npy"
print(f"\nTopography file exists: {topo_path.exists()}")
if topo_path.exists():
    topo = np.load(topo_path)
    print(f"Shape: {topo.shape}, min={topo.min():.4f}, max={topo.max():.4f}")

[START] model=random_forest paradigm=DS002680__gonogo_1 step=5
[DONE] model=random_forest paradigm=DS002680__gonogo_1 step=5 accuracy=0.5212 balanced_accuracy=0.5410 degenerate=0/10
Result: {'dataset_task': 'DS002680__gonogo_1', 'model': 'random_forest', 'step': '5', 'n_folds': 10, 'n_degenerate_folds': 0, 'accuracy_mean': 0.5211954165547386, 'balanced_accuracy_mean': 0.5410424947393032, 'macro_f1_mean': 0.3891621489553129, 'auc_mean': nan}

Topography file exists: True
Shape: (31,), min=0.0000, max=0.0003


In [31]:
# ============================================================
# Correct wrapper script using run_one() directly -- confirmed
# working function name and behavior this time
# ============================================================
wrapper_code = '''
import sys
sys.path.insert(0, "/lp-dev/23BRS1328/erp_trial_min")
import warnings
warnings.filterwarnings("ignore")
from stage4_classical_ml import run_stage1_classical_ml as driver

GONOGO_PARADIGMS = [f"DS002680__gonogo_{i}" for i in range(1, 14)]
STEPS = ["5", "10", "20", "30", "50", "75", "100", "150", "200", "full"]

for paradigm in GONOGO_PARADIGMS:
    print(f"\\n=== {paradigm} ===", flush=True)
    try:
        data = driver.ParadigmData(paradigm)
    except Exception as e:
        print(f"[ERROR] Could not load {paradigm}: {e}", flush=True)
        continue
    for model_name in ["random_forest", "xgboost"]:
        for step in STEPS:
            try:
                driver.run_one(model_name, paradigm, step, data)
            except Exception as e:
                print(f"[ERROR] {model_name}/{paradigm}/step={step}: {e}", flush=True)

print("\\nGonogo classical ML topography re-run complete.", flush=True)
'''

wrapper_path = project_root / "run_classical_gonogo_topo.py"
wrapper_path.write_text(wrapper_code)

log_path = project_root / "logs_classical_gonogo_topo2.log"
with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        ["python3", str(wrapper_path)],
        cwd=str(project_root), stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=1382887
Log: /lp-dev/23BRS1328/erp_trial_min/logs_classical_gonogo_topo2.log


In [32]:
import time
time.sleep(120)

result = subprocess.run(["ps", "-p", "1382887", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_classical_gonogo_topo2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l or "ERROR" in l]
print("\n".join(done_lines[-30:]))

# Progress check via saved topography files
topo_dir = project_root / "results_stage4_classical_ml" / "raw_topographies"
n_topos = len(list(topo_dir.rglob("*.npy"))) if topo_dir.exists() else 0
print(f"\nTopography files saved so far: {n_topos} / 260 expected")

=== Process status ===
    PID %CPU STAT     ELAPSED
1382887  197 Ssl        04:00

[DONE] model=xgboost paradigm=DS002680__gonogo_2 step=150 accuracy=0.5480 balanced_accuracy=0.5413 degenerate=0/11
[DONE] model=xgboost paradigm=DS002680__gonogo_2 step=200 accuracy=0.5480 balanced_accuracy=0.5413 degenerate=0/11
[DONE] model=xgboost paradigm=DS002680__gonogo_2 step=full accuracy=0.5699 balanced_accuracy=0.5705 degenerate=0/11
=== DS002680__gonogo_3 ===
[DONE] model=random_forest paradigm=DS002680__gonogo_3 step=5 accuracy=0.5365 balanced_accuracy=0.5365 degenerate=0/12
[DONE] model=random_forest paradigm=DS002680__gonogo_3 step=10 accuracy=0.5522 balanced_accuracy=0.5532 degenerate=0/12
[DONE] model=random_forest paradigm=DS002680__gonogo_3 step=20 accuracy=0.5474 balanced_accuracy=0.5485 degenerate=0/12
[DONE] model=random_forest paradigm=DS002680__gonogo_3 step=30 accuracy=0.5543 balanced_accuracy=0.5552 degenerate=0/12
[DONE] model=random_forest paradigm=DS002680__gonogo_3 step=50 a

In [33]:
import time
time.sleep(1800)  # 30 minutes

result = subprocess.run(["ps", "-p", "1382887", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_classical_gonogo_topo2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l or "ERROR" in l]
print("\n".join(done_lines[-20:]))

topo_dir = project_root / "results_stage4_classical_ml" / "raw_topographies"
n_topos = len(list(topo_dir.rglob("*.npy"))) if topo_dir.exists() else 0
print(f"\nTopography files saved so far: {n_topos} / 260 expected")

=== Process status ===
    PID %CPU STAT     ELAPSED
1382887 97.4 Zs         35:40

[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=5 accuracy=0.5401 balanced_accuracy=0.5798 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=10 accuracy=0.5445 balanced_accuracy=0.5862 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=20 accuracy=0.5475 balanced_accuracy=0.5897 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=30 accuracy=0.5477 balanced_accuracy=0.5896 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=50 accuracy=0.5501 balanced_accuracy=0.5920 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=75 accuracy=0.5501 balanced_accuracy=0.5920 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=100 accuracy=0.5501 balanced_accuracy=0.5920 degenerate=0/11
[DONE] model=random_forest paradigm=DS002680__gonogo_13 step=150 accu

In [34]:
# ============================================================
# Launch GAT on all 13 gonogo variants, full 10-step grid --
# same marker-based resumability, reusing the already-working
# run_gat_loso.py from tonight's main GAT work
# ============================================================
import dl_models.run_gat_loso as gat_loso
import importlib
importlib.reload(gat_loso)

wrapper_code = '''
import sys
sys.path.insert(0, "/lp-dev/23BRS1328/erp_trial_min")
import warnings
warnings.filterwarnings("ignore")
import torch
import dl_models.run_gat_loso as gat_loso

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}", flush=True)

GONOGO = [(f"DS002680_gonogo_{i}", "DS002680", f"gonogo_{i}") for i in range(1, 14)]

for paradigm_key, dataset, task in GONOGO:
    print(f"\\n=== {paradigm_key} ===", flush=True)
    for step in gat_loso.STEPS:
        try:
            gat_loso.run_paradigm_step(paradigm_key, dataset, task, step, device)
        except Exception as e:
            print(f"[ERROR] {paradigm_key} step={step}: {e}", flush=True)

print("\\nGAT gonogo re-run complete.", flush=True)
'''

wrapper_path = project_root / "run_gat_gonogo_full.py"
wrapper_path.write_text(wrapper_code)

log_path = project_root / "logs_gat_gonogo_full.log"
with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        ["python3", str(wrapper_path)],
        cwd=str(project_root), stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

/home/nvidia/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/nvidia/.venv/lib/python3.12/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSsb
  import torch_geometric.typing
/home/nvidia/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /home/nvidia/.venv/lib/python3.12/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSsb
  import torch_geometric.typing


Started, PID=1945152
Log: /lp-dev/23BRS1328/erp_trial_min/logs_gat_gonogo_full.log


In [35]:
import time
time.sleep(600)  # 10 minutes

result = subprocess.run(["ps", "-p", "1945152", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_gat_gonogo_full.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l or "ERROR" in l]
print("\n".join(done_lines[-25:]))

markers_gonogo = list((project_root / "results_gat" / "markers").glob("gat__DS002680_gonogo_*"))
print(f"\nGonogo GAT markers so far: {len(markers_gonogo)} / 130 expected")

=== Process status ===
    PID %CPU STAT     ELAPSED
1945152  168 Zs         11:54

[ERROR] DS002680_gonogo_11 step=150: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/cuda/CUDAContext.cpp":52, please report a bug to PyTorch. device=7, num_gpus=7
[ERROR] DS002680_gonogo_11 step=200: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/cuda/CUDAContext.cpp":52, please report a bug to PyTorch. device=7, num_gpus=7
[ERROR] DS002680_gonogo_11 step=full: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/cuda/CUDAContext.cpp":52, please report a bug to PyTorch. device=7, num_gpus=7
=== DS002680_gonogo_12 ===
[ERROR] DS002680_gonogo_12 step=5: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNA

In [36]:
# ============================================================
# Check current GPU state before relaunching -- confirm the
# device error was transient, not something that will repeat
# ============================================================
result = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], capture_output=True, text=True)
print(result.stdout)

# Also confirm from Python directly
import torch
print(f"\ntorch.cuda.device_count(): {torch.cuda.device_count()}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")

index, name, memory.used [MiB], memory.total [MiB]
0, NVIDIA A100-SXM4-80GB, 540 MiB, 81920 MiB
1, NVIDIA A100-SXM4-80GB, 685 MiB, 81920 MiB
2, NVIDIA A100-SXM4-80GB, 4 MiB, 81920 MiB
3, NVIDIA A100-SXM4-80GB, 44036 MiB, 81920 MiB
4, NVIDIA A100-SXM4-80GB, 36817 MiB, 81920 MiB
5, NVIDIA A100-SXM4-80GB, 4 MiB, 81920 MiB
6, NVIDIA A100-SXM4-80GB, 835 MiB, 81920 MiB
7, NVIDIA A100-SXM4-80GB, 70259 MiB, 81920 MiB


torch.cuda.device_count(): 8
torch.cuda.is_available(): True


DeferredCudaCallError: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/cuda/CUDAContext.cpp":52, please report a bug to PyTorch. device=7, num_gpus=7

CUDA call was originally invoked at:

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/nvidia/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/nvidia/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/nvidia/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/nvidia/.venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/home/nvidia/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/pyt

In [37]:
import subprocess
result = subprocess.run(
    ["python3", "-c",
     "import os; os.environ['CUDA_VISIBLE_DEVICES']='0,1,2,3,4,5,6'; "
     "import torch; print(f'device_count: {torch.cuda.device_count()}'); "
     "print(f'is_available: {torch.cuda.is_available()}'); "
     "x = torch.randn(10).cuda(); print(f'Test tensor device: {x.device}')"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1000:] if result.stderr else "(none)")

STDOUT: device_count: 7
is_available: True
Test tensor device: cuda:0

STDERR: /home/nvidia/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]



In [38]:
# ============================================================
# Relaunch GAT gonogo run with CUDA_VISIBLE_DEVICES fix applied,
# in the wrapper script itself so it's set before torch loads
# ============================================================
wrapper_code = '''
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6"

import sys
sys.path.insert(0, "/lp-dev/23BRS1328/erp_trial_min")
import warnings
warnings.filterwarnings("ignore")
import torch
import dl_models.run_gat_loso as gat_loso

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}", flush=True)
print(f"Visible GPU count: {torch.cuda.device_count()}", flush=True)

GONOGO = [(f"DS002680_gonogo_{i}", "DS002680", f"gonogo_{i}") for i in range(1, 14)]

for paradigm_key, dataset, task in GONOGO:
    print(f"\\n=== {paradigm_key} ===", flush=True)
    for step in gat_loso.STEPS:
        try:
            gat_loso.run_paradigm_step(paradigm_key, dataset, task, step, device)
        except Exception as e:
            print(f"[ERROR] {paradigm_key} step={step}: {e}", flush=True)

print("\\nGAT gonogo re-run complete.", flush=True)
'''

wrapper_path = project_root / "run_gat_gonogo_full.py"
wrapper_path.write_text(wrapper_code)

log_path = project_root / "logs_gat_gonogo_full2.log"
with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        ["python3", str(wrapper_path)],
        cwd=str(project_root), stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=2529385
Log: /lp-dev/23BRS1328/erp_trial_min/logs_gat_gonogo_full2.log


In [39]:
import time
time.sleep(300)  # 5 minutes -- quick check first to confirm no immediate crash

result = subprocess.run(["ps", "-p", "2529385", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_gat_gonogo_full2.log").read_text()
print("=== Log tail ===")
print(log_text[-1500:] if log_text else "(no output yet)")

markers_gonogo = list((project_root / "results_gat" / "markers").glob("gat__DS002680_gonogo_*"))
print(f"\nGonogo GAT markers so far: {len(markers_gonogo)} / 130 expected")

=== Process status ===
    PID %CPU STAT     ELAPSED
2529385 99.6 Rsl        05:30

=== Log tail ===
] model=gat paradigm=DS002680_gonogo_5 step=30 balanced_accuracy=0.5178 degenerate=0/12
[TIME] gat / DS002680_gonogo_5 step=30: 2.3s
[CACHED] DS002680_gonogo_5 already exists, skipping
[DONE] model=gat paradigm=DS002680_gonogo_5 step=50 balanced_accuracy=0.4727 degenerate=0/12
[TIME] gat / DS002680_gonogo_5 step=50: 3.2s
[CACHED] DS002680_gonogo_5 already exists, skipping
[DONE] model=gat paradigm=DS002680_gonogo_5 step=75 balanced_accuracy=0.5131 degenerate=0/12
[TIME] gat / DS002680_gonogo_5 step=75: 3.7s
[CACHED] DS002680_gonogo_5 already exists, skipping
[DONE] model=gat paradigm=DS002680_gonogo_5 step=100 balanced_accuracy=0.5329 degenerate=0/12
[TIME] gat / DS002680_gonogo_5 step=100: 5.1s
[CACHED] DS002680_gonogo_5 already exists, skipping
[DONE] model=gat paradigm=DS002680_gonogo_5 step=150 balanced_accuracy=0.4994 degenerate=0/12
[TIME] gat / DS002680_gonogo_5 step=150: 7.3s
[C

In [40]:
import time
time.sleep(600)  # 10 minutes

result = subprocess.run(["ps", "-p", "2529385", "-o", "pid,pcpu,stat,etime"], capture_output=True, text=True)
print("=== Process status ===")
print(result.stdout)

log_text = (project_root / "logs_gat_gonogo_full2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l or "ERROR" in l]
print("\n".join(done_lines[-20:]))

markers_gonogo = list((project_root / "results_gat" / "markers").glob("gat__DS002680_gonogo_*"))
print(f"\nGonogo GAT markers so far: {len(markers_gonogo)} / 130 expected")

=== Process status ===
    PID %CPU STAT     ELAPSED
2529385 90.4 Zs         17:32

[DONE] model=gat paradigm=DS002680_gonogo_12 step=10 balanced_accuracy=0.4896 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=20 balanced_accuracy=0.5163 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=30 balanced_accuracy=0.4995 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=50 balanced_accuracy=0.4815 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=75 balanced_accuracy=0.4758 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=100 balanced_accuracy=0.4877 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=150 balanced_accuracy=0.4954 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=200 balanced_accuracy=0.5136 degenerate=0/11
[DONE] model=gat paradigm=DS002680_gonogo_12 step=full balanced_accuracy=0.5074 degenerate=0/11
=== DS002680_gonogo_13 ===
[DONE] model=gat paradigm=DS002680_gon

In [41]:
# ============================================================
# 6.1 FINAL STEP: compute NFS for classical ML (RF, XGBoost)
# and GAT, against the validated pooled reference topography,
# across all 13 gonogo variants and all 10 steps
# ============================================================
from stage4_classical_ml.nfs import bootstrap_nfs

reference_topo = np.load(project_root / "results_gat" / "ds002680_pooled_reference_topography.npy")
STEP_ORDER = ["5", "10", "20", "30", "50", "75", "100", "150", "200", "full"]
GONOGO_PARADIGMS_CLASSICAL = [f"DS002680__gonogo_{i}" for i in range(1, 14)]
GONOGO_PARADIGMS_GAT = [f"DS002680_gonogo_{i}" for i in range(1, 14)]

all_nfs_rows = []

# --- Classical ML (RF, XGBoost) ---
for model_name in ["random_forest", "xgboost"]:
    for paradigm in GONOGO_PARADIGMS_CLASSICAL:
        for step in STEP_ORDER:
            topo_path = project_root / "results_stage4_classical_ml" / "raw_topographies" / model_name / paradigm / f"step_{step}.npy"
            if not topo_path.exists():
                continue
            topo_model = np.load(topo_path)
            if len(topo_model) != len(reference_topo):
                continue
            point, ci_low, ci_high = bootstrap_nfs(topo_model, reference_topo, n_iter=1000, seed=42)
            all_nfs_rows.append({
                "paradigm": paradigm, "model": model_name, "step": step,
                "nfs": point, "ci_low": ci_low, "ci_high": ci_high,
            })

# --- GAT ---
for paradigm in GONOGO_PARADIGMS_GAT:
    for step in STEP_ORDER:
        topo_path = project_root / "results_gat" / "attention_weights" / paradigm / f"step_{step}.npy"
        if not topo_path.exists():
            continue
        topo_model = np.load(topo_path)
        if len(topo_model) != len(reference_topo):
            continue
        point, ci_low, ci_high = bootstrap_nfs(topo_model, reference_topo, n_iter=1000, seed=42)
        all_nfs_rows.append({
            "paradigm": paradigm.replace("DS002680_", "DS002680__"), "model": "gat", "step": step,
            "nfs": point, "ci_low": ci_low, "ci_high": ci_high,
        })

nfs_6_1_df = pd.DataFrame(all_nfs_rows)
print(f"Total NFS values computed for 6.1: {len(nfs_6_1_df)}")
print(nfs_6_1_df.groupby("model")["nfs"].agg(["count", "mean", "min", "max"]))

out_path = project_root / "results_gat" / "ds002680_NFS_6_1.csv"
nfs_6_1_df.to_csv(out_path, index=False)
print(f"\n[Saved] {out_path}")

Total NFS values computed for 6.1: 388
               count      mean       min       max
model                                             
gat              128 -0.018282 -0.510105  0.573682
random_forest    130  0.105836 -0.429679  0.649618
xgboost          130  0.091351 -0.350872  0.553695

[Saved] /lp-dev/23BRS1328/erp_trial_min/results_gat/ds002680_NFS_6_1.csv
